# Tahap 4 — Case Solution Reuse (Prediksi Solusi)

Dari top-k kasus termirip, mengambil elemen putusan sebagai solusi atas kasus baru.

Algoritma prediksi:
1. **Majority Vote**: pilih solusi yang paling banyak muncul
2. **Weighted Similarity**: bobot = skor similarity^5

**Input**: `data/eval/queries.json`, `data/processed/cases.json`, model terlatih

**Output**: `data/results/predictions.csv`

## 4.1 Import Library

In [ ]:
import json
import pickle
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import AutoTokenizer, AutoModel

warnings.filterwarnings("ignore")

## 4.2 Load Data dan Model

In [ ]:
DATA_DIR   = Path("../data/processed")
EVAL_DIR   = Path("../data/eval")
MODEL_DIR  = Path("../models")
RESULT_DIR = Path("../data/results")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

with open(DATA_DIR / "cases.json", encoding="utf-8") as f:
    cases = json.load(f)
case_map = {c["case_id"]: c for c in cases}

with open(MODEL_DIR / "tfidf_vectorizer.pkl", "rb") as f:
    vectorizer = pickle.load(f)
all_embeddings = np.load(MODEL_DIR / "indobert_embeddings.npy")

texts = [c["text_full"] for c in cases]
all_vectors = vectorizer.transform(texts)

print(f"Loaded {len(cases)} kasus")
print(f"TF-IDF vectors: {all_vectors.shape}")
print(f"IndoBERT embeddings: {all_embeddings.shape}")

## 4.3 Load IndoBERT

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME = "indobenchmark/indobert-base-p1"
print(f"Loading IndoBERT: {MODEL_NAME} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModel.from_pretrained(MODEL_NAME).to(device)
bert_model.eval()
print(f"Device: {device}")

## 4.4 Fungsi Retrieval dan Prediksi

In [ ]:
def get_indobert_embedding(text, tokenizer, model, device, max_length=512):
    inputs = tokenizer(
        text, return_tensors="pt",
        truncation=True, max_length=max_length, padding=True
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    mask = inputs["attention_mask"].unsqueeze(-1).float()
    emb = (outputs.last_hidden_state * mask).sum(1) / mask.sum(1)
    return emb.cpu().numpy()[0]


def retrieve_tfidf(query_text, vectorizer, all_vectors, cases, k=5):
    q_vec = vectorizer.transform([query_text])
    sims = cosine_similarity(q_vec, all_vectors)[0]
    top_k = np.argsort(sims)[::-1][:k]
    return [(cases[i]["case_id"], sims[i]) for i in top_k]


def retrieve_bert(query_emb, all_embeddings, cases, k=5):
    q = query_emb.reshape(1, -1)
    sims = cosine_similarity(q, all_embeddings)[0]
    top_k = np.argsort(sims)[::-1][:k]
    return [(cases[i]["case_id"], sims[i]) for i in top_k]


def extract_solution(case):
    return {
        "label_pasal":     case.get("label_pasal", ""),
        "label_vonis":     case.get("label_vonis", ""),
        "amar_putusan":    case.get("amar_putusan", ""),
        "vonis_bulan":     case.get("vonis_bulan", 0),
        "ringkasan_fakta": case.get("ringkasan_fakta", "")[:300],
        "no_perkara":      case.get("no_perkara", ""),
    }


def majority_vote(solutions, key):
    values = [s[key] for s in solutions if s.get(key)]
    if not values:
        return "tidak_diketahui"
    return Counter(values).most_common(1)[0][0]


def weighted_vote(top_k_with_sim, case_map, key):
    scores = {}
    for cid, sim in top_k_with_sim:
        val = case_map[cid].get(key, "")
        if val:
            scores[val] = scores.get(val, 0.0) + (sim ** 5)
    if not scores:
        return "tidak_diketahui"
    return max(scores, key=scores.get)


def predict_vonis_bulan(top_k_with_sim, case_map, method="weighted"):
    if not top_k_with_sim:
        return 0
    if method == "majority":
        total_months = sum(case_map[cid].get("vonis_bulan", 0) for cid, _ in top_k_with_sim)
        return int(round(total_months / len(top_k_with_sim)))
    else:
        total_weight = sum((sim ** 5) for _, sim in top_k_with_sim)
        if total_weight == 0:
            return 0
        weighted_months = sum(case_map[cid].get("vonis_bulan", 0) * (sim ** 5) for cid, sim in top_k_with_sim)
        return int(round(weighted_months / total_weight))


def predict_outcome(query_text, k=5):
    tfidf_top_k = retrieve_tfidf(query_text, vectorizer, all_vectors, cases, k)
    tfidf_solutions = [extract_solution(case_map[cid]) for cid, _ in tfidf_top_k]
    tfidf_mv_pasal = majority_vote(tfidf_solutions, "label_pasal")
    tfidf_mv_vonis = predict_vonis_bulan(tfidf_top_k, case_map, "majority")
    tfidf_wv_pasal = weighted_vote(tfidf_top_k, case_map, "label_pasal")
    tfidf_wv_vonis = predict_vonis_bulan(tfidf_top_k, case_map, "weighted")

    query_emb = get_indobert_embedding(query_text, tokenizer, bert_model, device)
    bert_top_k = retrieve_bert(query_emb, all_embeddings, cases, k)
    bert_solutions = [extract_solution(case_map[cid]) for cid, _ in bert_top_k]
    bert_mv_pasal = majority_vote(bert_solutions, "label_pasal")
    bert_mv_vonis = predict_vonis_bulan(bert_top_k, case_map, "majority")
    bert_wv_pasal = weighted_vote(bert_top_k, case_map, "label_pasal")
    bert_wv_vonis = predict_vonis_bulan(bert_top_k, case_map, "weighted")

    return {
        "tfidf_top5":     [{"case_id": c, "sim": round(float(s), 4)} for c, s in tfidf_top_k],
        "tfidf_mv_pasal": tfidf_mv_pasal, "tfidf_mv_vonis": tfidf_mv_vonis,
        "tfidf_wv_pasal": tfidf_wv_pasal, "tfidf_wv_vonis": tfidf_wv_vonis,
        "bert_top5":      [{"case_id": c, "sim": round(float(s), 4)} for c, s in bert_top_k],
        "bert_mv_pasal":  bert_mv_pasal, "bert_mv_vonis": bert_mv_vonis,
        "bert_wv_pasal":  bert_wv_pasal, "bert_wv_vonis": bert_wv_vonis,
    }

## 4.5 Jalankan Prediksi pada Query Evaluasi

In [ ]:
with open(EVAL_DIR / "queries.json", encoding="utf-8") as f:
    queries = json.load(f)

rows = []
for q in queries:
    qid = q["query_id"]
    gt_pasal = q["ground_truth_label_pasal"]
    gt_vonis = q["ground_truth_label_vonis"]
    qtext = q["query_text"]

    result = predict_outcome(qtext, k=5)

    top5_tfidf_ids = "|".join(x["case_id"] for x in result["tfidf_top5"])
    top5_bert_ids = "|".join(x["case_id"] for x in result["bert_top5"])

    rows.append({
        "query_id": qid,
        "ground_truth_pasal": gt_pasal,
        "ground_truth_vonis": gt_vonis,
        "tfidf_mv_pasal": result["tfidf_mv_pasal"],
        "tfidf_mv_vonis": result["tfidf_mv_vonis"],
        "tfidf_wv_pasal": result["tfidf_wv_pasal"],
        "tfidf_wv_vonis": result["tfidf_wv_vonis"],
        "bert_mv_pasal":  result["bert_mv_pasal"],
        "bert_mv_vonis":  result["bert_mv_vonis"],
        "bert_wv_pasal":  result["bert_wv_pasal"],
        "bert_wv_vonis":  result["bert_wv_vonis"],
        "top5_tfidf_case_ids": top5_tfidf_ids,
        "top5_bert_case_ids":  top5_bert_ids,
    })

    print(f"  {qid}: GT_pasal={gt_pasal}, GT_vonis={gt_vonis}")
    print(f"         TF-IDF MV: pasal={result['tfidf_mv_pasal']}, vonis={result['tfidf_mv_vonis']}")
    print(f"         TF-IDF WV: pasal={result['tfidf_wv_pasal']}, vonis={result['tfidf_wv_vonis']}")
    print(f"         BERT   MV: pasal={result['bert_mv_pasal']}, vonis={result['bert_mv_vonis']}")
    print(f"         BERT   WV: pasal={result['bert_wv_pasal']}, vonis={result['bert_wv_vonis']}")
    print()

df = pd.DataFrame(rows)
out_path = RESULT_DIR / "predictions.csv"
df.to_csv(out_path, index=False)
print(f"\nPredictions disimpan: {out_path}")
df